# Paraboloidna slobodna površina u rotirajućem spremniku

**Poglavlje 4: Relativno mirovanje fluida**

Ovaj interaktivni prikaz nadopunjuje izvod oblika slobodne površine fluida u cilindričnom spremniku koji rotira oko vlastite osi konstantnom kutnom brzinom. Mijenjanjem kutne brzine, polumjera spremnika i početne visine fluida prati se ravnotežni paraboloid.

## Cilj

U cilindričnom spremniku koji rotira oko vlastite osi, slobodna površina fluida poprima oblik paraboloida. Centrifugalna sila u rotirajućem sustavu zajedno s gravitacijom daje efektivno polje sila koje određuje taj profil. Prikaz omogućuje:

1. mijenjanje kutne brzine $\omega$;
2. mijenjanje polumjera spremnika $R$;
3. mijenjanje početne visine fluida $h_0$ pri mirovanju;
4. praćenje visine fluida u središtu i na rubu spremnika.

## Pretpostavke modela

- spremnik rotira konstantnom kutnom brzinom $\omega$;
- fluid je dosegao stacionarno stanje u rotirajućem okviru;
- nema isparavanja, gubitaka, niti gubljenja fluida preko ruba;
- ukupni volumen fluida ostaje očuvan.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

U rotirajućem okviru efektivno ubrzanje u radijalnom smjeru iznosi $\omega^2 r$. Profil slobodne površine je paraboloid:

$$z(r) = z_d + \frac{\omega^2 r^2}{2g},$$

gdje je $z_d$ visina fluida u središtu (na osi rotacije). Iz očuvanja volumena:

$$z_d = h_0 - \frac{\omega^2 R^2}{4g}.$$

Visina fluida na rubu spremnika je $z_R = h_0 + \omega^2 R^2 / (4g)$. Kada $z_d < 0$, dno se ogoljava i dolazi do gubitka fluida preko ruba (zadaća se mijenja, pa ovaj model više ne vrijedi).

In [ ]:
G = 9.81

def paraboloid(omega, R_mm, h0_m):
    R = R_mm / 1000.0
    z_d = h0_m - (omega**2 * R**2) / (4 * G)
    z_R = h0_m + (omega**2 * R**2) / (4 * G)
    return {'z_d': z_d, 'z_R': z_R, 'R': R}

## Interaktivni prikaz

Klizačima u nastavku biraju se kutna brzina, polumjer spremnika i početna visina fluida pri mirovanju. Prikaz pokazuje aksijalni presjek spremnika s pripadnim paraboloidom.

In [ ]:
def paraboloid_prikaz(omega, R_mm, h0_m):
    r = paraboloid(omega, R_mm, h0_m)
    R = r['R']
    z_d = r['z_d']
    z_R = r['z_R']

    fig, ax = plt.subplots(figsize=(8, 6))

    # Profil paraboloida
    r_niz = np.linspace(-R, R, 200)
    z_niz = z_d + (omega**2 * r_niz**2) / (2 * G)

    # Visina spremnika (uzet ćemo 1.5 × h0)
    H_sprem = max(1.5 * h0_m, z_R + 0.1)

    # Stijenke spremnika
    ax.plot([-R, -R], [0, H_sprem], color='#444', lw=2.5)
    ax.plot([R, R], [0, H_sprem], color='#444', lw=2.5)
    ax.plot([-R, R], [0, 0], color='#444', lw=2.5)

    # Fluid (ispod paraboloida)
    if z_d >= 0:
        ax.fill_between(r_niz, 0, z_niz, fc='#aed6f1', alpha=0.7)
        upozorenje = ''
    else:
        # Dno je ogoljeno: fluid samo gdje je z_niz > 0
        z_mask = np.maximum(z_niz, 0)
        ax.fill_between(r_niz, 0, z_mask, fc='#aed6f1', alpha=0.7)
        upozorenje = '   (dno ogoljeno — model više ne vrijedi)'

    # Linija paraboloida
    ax.plot(r_niz, z_niz, color='#1565c0', lw=2.2,
             label='slobodna površina')

    # Linija početne razine pri mirovanju
    ax.axhline(h0_m, color='gray', ls='--', lw=1, alpha=0.7,
                label=f'$h_0$ = {h0_m:.2f} m (mirovanje)')

    # Oznake
    ax.annotate('', xy=(0, z_d), xytext=(0, h0_m),
                 arrowprops=dict(arrowstyle='<->', color='#c62828'))
    ax.text(0.02, (z_d + h0_m)/2,
             f'$h_0 - z_d$ = {h0_m - z_d:.3f} m',
             color='#c62828', fontsize=9)

    ax.set_xlim(-R*1.5, R*1.5)
    ax.set_ylim(-H_sprem*0.1, H_sprem*1.1)
    ax.set_xlabel('radijalna koordinata r (m)')
    ax.set_ylabel('visina z (m)')
    ax.set_title(
        f'$\\omega$ = {omega:.1f} rad/s,  $R$ = {R*1000:.0f} mm\n'
        f'$z_d$ = {z_d:.3f} m,  $z_R$ = {z_R:.3f} m{upozorenje}'
    )
    ax.legend(loc='upper center', fontsize=9)
    ax.grid(ls=':', alpha=0.5)

    plt.tight_layout()
    plt.show()


interact(
    paraboloid_prikaz,
    omega=FloatSlider(min=0, max=20, step=0.2, value=8,
                       description='$\\omega$ (rad/s)',
                       layout=Layout(width='420px')),
    R_mm=FloatSlider(min=50, max=400, step=10, value=150,
                      description='$R$ (mm)',
                      layout=Layout(width='420px')),
    h0_m=FloatSlider(min=0.1, max=0.8, step=0.02, value=0.30,
                      description='$h_0$ (m)',
                      layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Granica ogoljavanja dna.** Pri kojoj kombinaciji $\omega$ i $R$ središte spremnika počinje ogoljavati ($z_d = 0$)? Izvodom iz formule pokaži tu vezu.

2. **Skala s polumjerom.** Ako se polumjer spremnika udvostruči uz konstantne $\omega$ i $h_0$, kako se mijenja razlika $z_R - z_d$? Koji je eksponent ovisnosti?

3. **Volumno očuvanje.** Provjeri za nekoliko kombinacija da je volumen paraboloida iznad osnovne razine $h_0$ jednak volumenu praznog prostora ispod te razine u središtu spremnika.

4. **Tehnička primjena.** Centrifuga laboratorijskog stripa rotira pri $\omega \approx 100$ rad/s. Koliki bi bio paraboloid u cijevi polumjera $10$ mm? Što to govori o pretpostavkama ovog modela u stvarnoj centrifugi?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira ravnotežu u rotirajućem fluidu iz poglavlja 4. Paraboloidni oblik slobodne površine izravna je posljedica linearne ovisnosti centrifugalnog ubrzanja o radijalnoj udaljenosti. Isti se model koristi pri analizi centrifuga, rotirajućih spremnika za miješanje, ali i pri procjeni ponašanja goriva u spremniku autocisterne u zavoju.